# MCQ Generation — Step 1: File Loading and Preprocessing

Loads all `_v1_chunks.json` files into a unified corpus, merges figure captions into
parent chunks, filters out reference/stub/figure-only chunks, and builds an enriched
context string (section path prepended) for each surviving chunk.

In [ ]:
import glob
import json
import re
from dataclasses import dataclass, field
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_GLOB = str(REPO_ROOT / "output" / "*" / "auto" / "*_v1_chunks.json")

MIN_CHUNK_CHARS = 100
EXCLUDED_SECTION_KEYWORDS = ("REFERENCES", "BIBLIOGRAPHY")
FIGURE_PLACEHOLDER_RE = re.compile(r">\s*\[FIGURE:[^\]]+\]")

## Load raw chunk files

Each `_v1_chunks.json` is `{doc_id, source, chunks, figures}`. `figures[i].referenced_from`
points at the `chunk_id` it belongs to — that's the join key for the caption merge.

In [ ]:
@dataclass
class Chunk:
    uid: str
    doc_id: str
    chunk_id: str
    section_path: list[str]
    text: str
    context: str = field(default="")


def load_chunk_files(pattern: str = CHUNKS_GLOB) -> list[dict]:
    paths = sorted(glob.glob(pattern))
    assert paths, f"no chunk files matched {pattern!r}"
    docs = [json.load(open(p, encoding="utf-8")) for p in paths]
    print(f"loaded {len(docs)} doc(s) from {len(paths)} file(s)")
    return docs


raw_docs = load_chunk_files()

## Merge figure captions into parent chunks

Note: in this corpus the captioning pipeline already inlines `[FIGURE:fig_id] <caption>`
into chunk text (see `src/ingestion/upload_visuals.py` / `merge_by_page.py`). This step
is still applied defensively via `referenced_from` so the pipeline is correct even against
chunk files where captions were *not* pre-merged upstream — appending is a no-op when the
caption text is already a substring of the chunk body.

In [ ]:
def merge_captions(doc: dict) -> dict[str, str]:
    """Return chunk_id -> text, with any not-yet-inlined figure captions appended."""
    captions_by_chunk: dict[str, list[str]] = {}
    for fig in doc.get("figures", []):
        parent = fig.get("referenced_from")
        caption = fig.get("caption", "").strip()
        if parent and caption:
            captions_by_chunk.setdefault(parent, []).append(caption)

    merged: dict[str, str] = {}
    for chunk in doc["chunks"]:
        text = chunk["text"]
        for caption in captions_by_chunk.get(chunk["chunk_id"], []):
            if caption not in text:
                text = f"{text}\n\n{caption}"
        merged[chunk["chunk_id"]] = text
    return merged

## Filter unusable chunks

Drops: reference/bibliography sections, sub-100-char stub chunks, and chunks that are
purely a figure placeholder with no surrounding prose.

In [ ]:
def is_reference_section(section_path: list[str]) -> bool:
    joined = " ".join(section_path).upper()
    return any(kw in joined for kw in EXCLUDED_SECTION_KEYWORDS)


def is_figure_placeholder_only(text: str) -> bool:
    return not FIGURE_PLACEHOLDER_RE.sub("", text).strip()


def should_keep(section_path: list[str], text: str) -> bool:
    if is_reference_section(section_path):
        return False
    if len(text) < MIN_CHUNK_CHARS:
        return False
    if is_figure_placeholder_only(text):
        return False
    return True

## Build enriched context (section path as topic anchor)

The most specific heading — the last element of `section_path` — is prepended so the
LLM knows the governing concept before reading bullet-fragment body text.

In [ ]:
def build_context(section_path: list[str], text: str) -> str:
    topic = section_path[-1] if section_path else ""
    return f"{topic}\n\n{text}" if topic else text

## Assemble the unified corpus

In [ ]:
def build_corpus(raw_docs: list[dict]) -> list[Chunk]:
    corpus: list[Chunk] = []
    dropped = 0
    for doc in raw_docs:
        doc_id = doc["doc_id"]
        merged_text_by_id = merge_captions(doc)
        for chunk in doc["chunks"]:
            chunk_id = chunk["chunk_id"]
            section_path = chunk["section_path"]
            text = merged_text_by_id[chunk_id]

            if not should_keep(section_path, text):
                dropped += 1
                continue

            corpus.append(
                Chunk(
                    uid=f"{doc_id}::{chunk_id}",
                    doc_id=doc_id,
                    chunk_id=chunk_id,
                    section_path=section_path,
                    text=text,
                    context=build_context(section_path, text),
                )
            )
    print(f"kept {len(corpus)} chunk(s), dropped {dropped}")
    return corpus


corpus = build_corpus(raw_docs)

In [ ]:
# Sanity check — inspect a few surviving chunks
for chunk in corpus[:3]:
    print(chunk.uid)
    print(chunk.context[:300])
    print("---")

# Step 2 — Triple Extraction from Each Chunk

For each enriched chunk, a single Qwen3.5-27B prompt extracts candidate `{subject, relation, object}`
triples and assigns a difficulty tier per triple, in one pass. Relations are constrained to a
fixed taxonomy so distractor selection later has a closed vocabulary to work with.

Difficulty tiers:
- **1 (recall)** — a direct named relationship between two entities.
- **2 (application)** — involves a mechanism or process.
- **3 (reasoning)** — requires connecting this triple with another fact from the same chunk.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, field_validator

RELATION_TAXONOMY = (
    "PRODUCES",
    "REGULATES",
    "INHIBITS",
    "STIMULATES",
    "ACTS_ON",
    "PART_OF",
    "LOCATED_IN",
    "INNERVATES",
    "SUPPLIES",
    "CAUSES",
    "DIAGNOSED_BY",
    "TREATED_BY",
    "PREREQUISITE_OF",
)

Relation = Literal[
    "PRODUCES",
    "REGULATES",
    "INHIBITS",
    "STIMULATES",
    "ACTS_ON",
    "PART_OF",
    "LOCATED_IN",
    "INNERVATES",
    "SUPPLIES",
    "CAUSES",
    "DIAGNOSED_BY",
    "TREATED_BY",
    "PREREQUISITE_OF",
]

DifficultyTier = Literal[1, 2, 3]


class Triple(BaseModel):
    subject: str = Field(min_length=1)
    relation: Relation
    object: str = Field(min_length=1)
    difficulty: DifficultyTier


class TripleExtractionResult(BaseModel):
    triples: list[Triple]

In [ ]:
# Qwen3.5-27B triple extractor: reads a chunk's enriched context and emits a
# constrained-taxonomy relation triple list with per-triple difficulty tier, in one pass.
#
# Loaded lazily and released after use, matching the captioning module's memory-management
# pattern (src/captioning/qwen_vl.py) — same unified memory budget on MPS, only one
# model resident at a time.
#
# Qwen3.5 is a multimodal checkpoint (AutoModelForMultimodalLM + AutoProcessor rather than
# AutoModelForCausalLM + AutoTokenizer) but runs fine text-only — just omit image content
# from the messages.
#
# enable_thinking=False: this is structured extraction, not open-ended reasoning — thinking
# mode burns tokens producing a chain-of-thought this task doesn't need and complicates
# parsing the JSON back out.

import gc
import json
import logging
import re

import torch
from pydantic import ValidationError
from transformers import AutoModelForMultimodalLM, AutoProcessor

MODEL_PATH = "Qwen/Qwen3.5-27B"

log = logging.getLogger("triple_extractor")

_SYSTEM_PROMPT = f"""You are a medical knowledge graph extractor. Read the given passage and \
extract factual relationship triples between named medical entities (organs, hormones, \
enzymes, structures, diseases, processes, etc).

Each triple must use exactly one relation from this fixed taxonomy — do not invent new \
relations, do not use synonyms, only these:
{", ".join(RELATION_TAXONOMY)}

Never use a passive/inverse form of a relation (e.g. "STIMULATED_BY", "REGULATED_BY", \
"INHIBITED_BY", "SYNTHESIZED_BY", "DEGRADED_BY" are all forbidden). If the natural \
direction of a fact is passive, flip subject and object instead so the active relation \
from the taxonomy applies — e.g. "TSH is stimulated by TRH" becomes \
{{"subject": "TRH", "relation": "STIMULATES", "object": "TSH"}}, not \
{{"subject": "TSH", "relation": "STIMULATED_BY", "object": "TRH"}}.

For each triple, assign a difficulty tier:
- 1 (recall): a direct named relationship between two entities, statable from a single fact.
- 2 (application): involves a mechanism or process, not just a bare fact.
- 3 (reasoning): answering it requires connecting this triple with at least one other fact \
from the same passage.

Only extract triples that are explicitly supported by the passage text. Do not infer outside \
medical knowledge that isn't stated. If no valid triples exist, return an empty list.

Keep subject/object entity names short (a few words) — do not include full descriptive \
clauses in an entity name.

Respond with ONLY a JSON object of the form:
{{"triples": [{{"subject": "...", "relation": "...", "object": "...", "difficulty": 1}}]}}
No prose, no markdown fences, no explanation — JSON only."""

_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def _repair_truncated_json(raw: str) -> str:
    """Best-effort fix for JSON cut off mid-generation (max_new_tokens truncation): drops
    back to the last fully-formed value (closing an unterminated string first if needed),
    then closes whatever braces/brackets are still open — in the correct nesting order —
    so json.loads has a chance at the still-complete triples."""
    text = raw[raw.index("{"):] if "{" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    last_safe = max(text.rfind("}"), text.rfind("]"), text.rfind(","))
    if last_safe == -1:
        raise ValueError("no safe truncation point found")
    text = text[:last_safe] if text[last_safe] == "," else text[: last_safe + 1]

    # walk the (string-aware) delimiter stack to close whatever's still open, innermost first
    stack: list[str] = []
    in_string = False
    escape = False
    for ch in text:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch in "{[":
            stack.append(ch)
        elif ch in "}]":
            stack.pop()

    closers = {"{": "}", "[": "]"}
    return text + "".join(closers[ch] for ch in reversed(stack))


class TripleExtractor:
    def __init__(self, model_path: str = MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            return
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        self.model = AutoModelForMultimodalLM.from_pretrained(
            self.model_path,
            dtype=torch.bfloat16,
        )
        self.model.to(self.device)
        self.model.eval()

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def extract(self, context: str, max_new_tokens: int = 2048, max_retries: int = 2) -> TripleExtractionResult:
        """Retries only if EVERY triple in a response is invalid (i.e. the model produced
        unusable JSON or nothing worth keeping) — a response with some good and some bad
        triples keeps the good ones rather than discarding the whole batch."""
        assert self.model is not None, "call load() first"
        last_error: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(context, max_new_tokens=max_new_tokens)
            try:
                result = self._parse(raw)
            except ValueError as e:
                last_error = e
                log.warning("extraction parse failed (attempt %d/%d): %s", attempt + 1, max_retries + 1, e)
                log.warning("raw output was: %r", raw)
                continue
            if result.triples:
                return result
            last_error = ValueError("no valid triples in response")
            log.warning("extraction yielded 0 valid triples (attempt %d/%d)", attempt + 1, max_retries + 1)
        assert last_error is not None
        raise last_error

    def _parse(self, raw: str) -> TripleExtractionResult:
        # Qwen3.5 is a reasoning model — strip a <think>...</think> block if the
        # thinking-disable chat template kwarg didn't take, so stray braces inside the
        # reasoning trace don't get matched instead of the real JSON answer.
        cleaned = _THINK_BLOCK_RE.sub("", raw)

        matches = _JSON_BLOCK_RE.findall(cleaned)
        if not matches:
            raise ValueError(f"no JSON object found in model output: {raw[:200]!r}")
        candidate = matches[-1]

        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))

        raw_triples = payload.get("triples", [])
        triples: list[Triple] = []
        for item in raw_triples:
            try:
                triples.append(Triple.model_validate(item))
            except ValidationError as e:
                log.warning("dropping invalid triple %r: %s", item, e)
        return TripleExtractionResult(triples=triples)

    def _generate(self, context: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": _SYSTEM_PROMPT},
            {"role": "user", "content": context},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


extractor = TripleExtractor()
extractor.load()

In [ ]:
from dataclasses import dataclass

from tqdm.auto import tqdm


@dataclass
class ExtractedChunk:
    chunk: "Chunk"
    triples: list[Triple]


def extract_triples_for_corpus(corpus: list, extractor: TripleExtractor) -> list[ExtractedChunk]:
    results: list[ExtractedChunk] = []
    failures = 0
    for chunk in tqdm(corpus, desc="extracting triples"):
        try:
            result = extractor.extract(chunk.context)
        except Exception as e:
            failures += 1
            tqdm.write(f"skipping {chunk.uid}: {e}")
            continue
        results.append(ExtractedChunk(chunk=chunk, triples=result.triples))
    print(f"extracted triples for {len(results)}/{len(corpus)} chunk(s), {failures} failure(s)")
    return results

In [ ]:
# Smoke test on a small sample before running the full corpus (slow on local inference) —
# bump SAMPLE_SIZE to None to run everything.
SAMPLE_SIZE = 5
sample = corpus[:SAMPLE_SIZE] if SAMPLE_SIZE else corpus

extracted = extract_triples_for_corpus(sample, extractor)

In [ ]:
# Sanity check — inspect extracted triples
for ec in extracted:
    print(ec.chunk.uid)
    for t in ec.triples:
        print(f"  ({t.subject}) -[{t.relation}]-> ({t.object})  tier={t.difficulty}")
    print("---")

# Step 3 — Distractor Generation from the Corpus

Without a graph, distractor selection is a corpus-level retrieval problem. For each triple's
correct answer (its object), find plausible-but-wrong alternatives in two tiers:

- **Tier 1 — same-section entity retrieval.** Group all triples by the chunk's top-level
  `section_path` topic. For a given triple, candidate distractors are objects of *other*
  triples sharing the same relation within that same section (a same-type proxy — no explicit
  entity typing exists yet, so "same relation, same section" stands in for "same kind of
  entity"). E.g. other `REGULATES`-object hormones mentioned in the Parathyroid section.
- **Tier 2 — LLM-generated fallback.** If tier 1 doesn't yield 3 candidates, ask the LLM for
  the remainder: given the correct answer, relation, and topic context, generate distractors
  that are medically plausible but wrong — preferring entities that already appear elsewhere
  in the corpus over freely invented ones.

This is simpler than a full KG walk but adequate for a prototype baseline.

## Tier 1 — build the same-section, same-relation entity index

In [ ]:
from collections import defaultdict


def top_level_topic(section_path: list[str]) -> str:
    return section_path[0] if section_path else "UNSECTIONED"


def build_entity_index(extracted: list[ExtractedChunk]) -> dict[tuple[str, str], set[str]]:
    """(topic, relation) -> set of object entity names seen anywhere in that section."""
    index: dict[tuple[str, str], set[str]] = defaultdict(set)
    for ec in extracted:
        topic = top_level_topic(ec.chunk.section_path)
        for t in ec.triples:
            index[(topic, t.relation)].add(t.object)
    return index


def all_corpus_objects(extracted: list[ExtractedChunk]) -> list[str]:
    """Flat, deduped list of every object entity in the corpus — used as a vocabulary hint
    for the tier-2 LLM fallback, so invented distractors still lean on real corpus terms."""
    seen: set[str] = set()
    for ec in extracted:
        for t in ec.triples:
            seen.add(t.object)
    return sorted(seen)


entity_index = build_entity_index(extracted)
corpus_objects = all_corpus_objects(extracted)
print(f"entity index covers {len(entity_index)} (topic, relation) pair(s), {len(corpus_objects)} unique object(s)")

## Tier 1 lookup + Tier 2 LLM fallback

In [ ]:
import random

NUM_DISTRACTORS = 3


class DistractorSet(BaseModel):
    correct_answer: str
    distractors: list[str] = Field(min_length=3, max_length=3)
    sources: list[str] = Field(min_length=3, max_length=3)  # "same_section" | "llm", parallel to distractors

    @field_validator("distractors")
    @classmethod
    def distractors_differ_from_correct_answer(cls, v: list[str], info) -> list[str]:
        correct = info.data.get("correct_answer", "")
        if any(d.strip().lower() == correct.strip().lower() for d in v):
            raise ValueError("a distractor duplicates the correct answer")
        return v


def tier1_candidates(
    triple: Triple, topic: str, entity_index: dict[tuple[str, str], set[str]], k: int = NUM_DISTRACTORS
) -> list[str]:
    pool = entity_index.get((topic, triple.relation), set())
    candidates = [obj for obj in pool if obj.strip().lower() != triple.object.strip().lower()]
    random.shuffle(candidates)
    return candidates[:k]

In [ ]:
# Tier-2 fallback reuses the already-loaded extractor's model/processor rather than loading
# a second copy — same unified-memory budget constraint as the triple extractor.

_DISTRACTOR_SYSTEM_PROMPT = """You are a medical education question writer generating \
multiple-choice distractors. Given a correct answer, the relation it was derived from, and \
a topic, generate exactly {n} medically plausible but factually WRONG alternatives for this \
specific fact.

Prefer distractors from this list of terms that already appear elsewhere in the corpus over \
freely invented ones — they should match the vocabulary and difficulty level of the material:
{vocab_hint}

Respond with ONLY a JSON object of the form:
{{"distractors": ["...", "..."]}}
No prose, no markdown fences, no explanation — JSON only."""


class DistractorGenerator:
    def __init__(self, model, processor, device: str):
        self.model = model
        self.processor = processor
        self.device = device

    def generate(self, correct_answer: str, relation: str, topic: str, vocab_hint: list[str], n: int) -> list[str]:
        if n <= 0:
            return []
        prompt = (
            f"Correct answer: {correct_answer}\nRelation: {relation}\nTopic: {topic}\n"
            f"Generate exactly {n} distractor(s)."
        )
        system = _DISTRACTOR_SYSTEM_PROMPT.format(
            n=n, vocab_hint=", ".join(random.sample(vocab_hint, min(20, len(vocab_hint))))
        )
        raw = self._generate(system, prompt, max_new_tokens=256)
        return self._parse(raw, n)

    def _parse(self, raw: str, n: int) -> list[str]:
        cleaned = _THINK_BLOCK_RE.sub("", raw)
        matches = _JSON_BLOCK_RE.findall(cleaned)
        if not matches:
            raise ValueError(f"no JSON object found in distractor output: {raw[:200]!r}")
        candidate = matches[-1]
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))
        distractors = [d for d in payload.get("distractors", []) if isinstance(d, str) and d.strip()]
        return distractors[:n]

    def _generate(self, system: str, user: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


distractor_generator = DistractorGenerator(extractor.model, extractor.processor, extractor.device)

## Combine tiers and generate distractors for every triple

In [ ]:
@dataclass
class TripleWithDistractors:
    chunk: "Chunk"
    triple: Triple
    distractor_set: DistractorSet


def build_distractors_for_triple(
    triple: Triple,
    topic: str,
    entity_index: dict[tuple[str, str], set[str]],
    corpus_objects: list[str],
    distractor_generator: DistractorGenerator,
) -> DistractorSet | None:
    tier1 = tier1_candidates(triple, topic, entity_index)
    distractors = list(tier1)
    sources = ["same_section"] * len(tier1)

    remainder = NUM_DISTRACTORS - len(distractors)
    if remainder > 0:
        try:
            llm_distractors = distractor_generator.generate(
                correct_answer=triple.object,
                relation=triple.relation,
                topic=topic,
                vocab_hint=corpus_objects,
                n=remainder,
            )
        except (ValueError, json.JSONDecodeError) as e:
            log.warning("tier-2 distractor generation failed for %r: %s", triple, e)
            return None
        distractors.extend(llm_distractors)
        sources.extend(["llm"] * len(llm_distractors))

    if len(distractors) < NUM_DISTRACTORS:
        return None

    try:
        return DistractorSet(correct_answer=triple.object, distractors=distractors, sources=sources)
    except ValidationError as e:
        log.warning("invalid distractor set for %r: %s", triple, e)
        return None


def generate_distractors_for_corpus(
    extracted: list[ExtractedChunk],
    entity_index: dict[tuple[str, str], set[str]],
    corpus_objects: list[str],
    distractor_generator: DistractorGenerator,
) -> list[TripleWithDistractors]:
    results: list[TripleWithDistractors] = []
    skipped = 0
    all_triples = [(ec.chunk, t) for ec in extracted for t in ec.triples]
    for chunk, triple in tqdm(all_triples, desc="generating distractors"):
        topic = top_level_topic(chunk.section_path)
        distractor_set = build_distractors_for_triple(triple, topic, entity_index, corpus_objects, distractor_generator)
        if distractor_set is None:
            skipped += 1
            continue
        results.append(TripleWithDistractors(chunk=chunk, triple=triple, distractor_set=distractor_set))
    print(f"built distractors for {len(results)}/{len(all_triples)} triple(s), {skipped} skipped")
    return results

In [ ]:
triples_with_distractors = generate_distractors_for_corpus(extracted, entity_index, corpus_objects, distractor_generator)

In [ ]:
# Sanity check — inspect built distractor sets
for twd in triples_with_distractors[:5]:
    t, ds = twd.triple, twd.distractor_set
    print(f"({t.subject}) -[{t.relation}]-> ({t.object})  tier={t.difficulty}")
    print(f"  correct: {ds.correct_answer}")
    for d, src in zip(ds.distractors, ds.sources):
        print(f"  distractor ({src}): {d}")
    print("---")

# Step 4 — Question Stem Generation

For each triple and its three distractors, generate the question stem in a single LLM call.
The prompt passes the triple, the difficulty tier, the chunk's section path as topic context,
and the four answer options (one correct, three distractors) — the LLM outputs a question
stem only, not a full MCQ with labelled options, since option formatting happens in the
application layer.

Tier shapes the stem:
- **Tier 1** — a direct factual question.
- **Tier 2** — a mechanism or process question.
- **Tier 3** — a short clinical scenario/vignette using the section's topic as context.

Constraint: the stem must not contain the correct answer as a word or obvious synonym. This
is a prompt-level instruction only — no validation loop yet to enforce it.

In [ ]:
_TIER_INSTRUCTIONS = {
    1: "This is a tier 1 (recall) fact: write a direct factual question stem.",
    2: "This is a tier 2 (application) fact: write a stem about the mechanism or process involved.",
    3: "This is a tier 3 (reasoning) fact: write a short clinical scenario or case vignette that "
    "uses the topic as its clinical context, requiring the reader to connect this fact to answer.",
}

_STEM_SYSTEM_PROMPT = """You are a medical education question writer. Given a factual triple \
(subject, relation, object), its difficulty tier, a topic, and four answer options, write ONE \
multiple-choice question stem that the object is the correct answer to.

{tier_instruction}

Hard constraint: the stem must NOT contain the correct answer text, or an obvious synonym of \
it, anywhere in the stem — the reader must be able to select it only by knowing the fact, not \
by pattern-matching the stem.

Do not output the options or indicate which one is correct — only the question stem itself.

Respond with ONLY a JSON object of the form:
{{"stem": "..."}}
No prose, no markdown fences, no explanation — JSON only."""


class StemGenerator:
    def __init__(self, model, processor, device: str):
        self.model = model
        self.processor = processor
        self.device = device

    def generate(self, triple: Triple, topic: str, options: list[str], max_retries: int = 2) -> str | None:
        system = _STEM_SYSTEM_PROMPT.format(tier_instruction=_TIER_INSTRUCTIONS[triple.difficulty])
        user = (
            f"Subject: {triple.subject}\nRelation: {triple.relation}\nObject (correct answer): {triple.object}\n"
            f"Topic: {topic}\nOptions: {', '.join(options)}"
        )
        last_error: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(system, user, max_new_tokens=256)
            try:
                stem = self._parse(raw)
            except ValueError as e:
                last_error = e
                log.warning("stem parse failed (attempt %d/%d): %s", attempt + 1, max_retries + 1, e)
                log.warning("raw output was: %r", raw)
                continue
            if triple.object.strip().lower() in stem.lower():
                last_error = ValueError("stem leaks the correct answer")
                log.warning("stem leaked correct answer (attempt %d/%d): %r", attempt + 1, max_retries + 1, stem)
                continue
            return stem
        log.warning("giving up on stem after %d attempts: %s", max_retries + 1, last_error)
        return None

    def _parse(self, raw: str) -> str:
        cleaned = _THINK_BLOCK_RE.sub("", raw)
        matches = _JSON_BLOCK_RE.findall(cleaned)
        if not matches:
            raise ValueError(f"no JSON object found in stem output: {raw[:200]!r}")
        candidate = matches[-1]
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))
        stem = payload.get("stem", "").strip()
        if not stem:
            raise ValueError("empty stem")
        return stem

    def _generate(self, system: str, user: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


stem_generator = StemGenerator(extractor.model, extractor.processor, extractor.device)

## Generate stems for all distractor sets

In [ ]:
@dataclass
class MCQItem:
    uid: str
    doc_id: str
    chunk_id: str
    topic: str
    section_path: list[str]
    source_context: str
    triple: Triple
    stem: str
    correct_answer: str
    distractors: list[str]
    distractor_sources: list[str]


def generate_mcq_items(triples_with_distractors: list[TripleWithDistractors], stem_generator: StemGenerator) -> list[MCQItem]:
    results: list[MCQItem] = []
    skipped = 0
    for i, twd in enumerate(tqdm(triples_with_distractors, desc="generating stems")):
        chunk, triple, ds = twd.chunk, twd.triple, twd.distractor_set
        topic = top_level_topic(chunk.section_path)
        options = [ds.correct_answer, *ds.distractors]
        random.shuffle(options)

        stem = stem_generator.generate(triple, topic, options)
        if stem is None:
            skipped += 1
            continue

        results.append(
            MCQItem(
                uid=f"{chunk.uid}::t{i:04d}",
                doc_id=chunk.doc_id,
                chunk_id=chunk.chunk_id,
                topic=topic,
                section_path=chunk.section_path,
                source_context=chunk.context,
                triple=triple,
                stem=stem,
                correct_answer=ds.correct_answer,
                distractors=ds.distractors,
                distractor_sources=ds.sources,
            )
        )
    print(f"generated {len(results)}/{len(triples_with_distractors)} MCQ item(s), {skipped} skipped/failed")
    return results


mcq_items = generate_mcq_items(triples_with_distractors, stem_generator)

In [ ]:
# Step 5 — Self-Critique Validation Loop

A second LLM call acts as critic on the full generated MCQ (stem, correct answer, three
distractors), checking four things against the source chunk text:
1. Is the stem unambiguous?
2. Is the correct answer definitively correct given the source text?
3. Could any distractor be argued correct?
4. Does the difficulty tier match the question's actual cognitive demand?

The critic returns a binary valid/invalid verdict plus a brief reason if invalid. If invalid,
a third LLM call corrects the specific problem identified. Capped at one critique-and-correct
cycle for this prototype (no iterating until valid) — this is MCQG-SRefine simplified.

## Critic — checks the full MCQ against the source passage

In [ ]:
class Critique(BaseModel):
    valid: bool
    reason: str | None = None


_CRITIC_SYSTEM_PROMPT = """You are a rigorous medical education reviewer. You will be given a \
source passage and a multiple-choice question (stem, correct answer, three distractors) \
generated from it. Check all four of the following:

1. Is the stem unambiguous — does it have exactly one reasonable interpretation?
2. Is the correct answer definitively correct, explicitly stated or directly supported by the \
source passage?
3. Could any of the three distractors be reasonably argued as also correct, given the source \
passage?
4. Does the stated difficulty tier (1=recall, 2=application, 3=reasoning) match the question's \
actual cognitive demand?

If ALL four checks pass, the question is valid. If ANY check fails, it is invalid — give a \
brief reason naming which check failed and why.

Respond with ONLY a JSON object of the form:
{"valid": true} or {"valid": false, "reason": "..."}
No prose, no markdown fences, no explanation outside the JSON — JSON only."""


class Critic:
    def __init__(self, model, processor, device: str):
        self.model = model
        self.processor = processor
        self.device = device

    def critique(self, item: "MCQItem", max_retries: int = 2) -> Critique:
        """Defaults to valid=True if the critic itself can't be parsed after retries — a
        broken critic call shouldn't silently drop otherwise-fine questions."""
        options = [item.correct_answer, *item.distractors]
        user = (
            f"Source passage:\n{item.source_context}\n\n"
            f"Question stem: {item.stem}\n"
            f"Correct answer: {item.correct_answer}\n"
            f"Distractors: {', '.join(d for d in options if d != item.correct_answer)}\n"
            f"Stated difficulty tier: {item.triple.difficulty}"
        )
        last_error: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(user, max_new_tokens=256)
            try:
                return self._parse(raw)
            except ValueError as e:
                last_error = e
                log.warning("critique parse failed (attempt %d/%d): %s", attempt + 1, max_retries + 1, e)
                log.warning("raw output was: %r", raw)
        log.warning("giving up on critique after %d attempts: %s — defaulting to valid", max_retries + 1, last_error)
        return Critique(valid=True)

    def _parse(self, raw: str) -> Critique:
        cleaned = _THINK_BLOCK_RE.sub("", raw)
        matches = _JSON_BLOCK_RE.findall(cleaned)
        if not matches:
            raise ValueError(f"no JSON object found in critique output: {raw[:200]!r}")
        candidate = matches[-1]
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))
        return Critique.model_validate(payload)

    def _generate(self, user: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": _CRITIC_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


critic = Critic(extractor.model, extractor.processor, extractor.device)

## Corrector — fixes the specific problem the critic flagged

In [ ]:
class CorrectedMCQ(BaseModel):
    stem: str = Field(min_length=1)
    correct_answer: str = Field(min_length=1)
    distractors: list[str] = Field(min_length=3, max_length=3)
    difficulty: DifficultyTier

    @field_validator("distractors")
    @classmethod
    def distractors_differ_from_correct_answer(cls, v: list[str], info) -> list[str]:
        correct = info.data.get("correct_answer", "")
        if any(d.strip().lower() == correct.strip().lower() for d in v):
            raise ValueError("a distractor duplicates the correct answer")
        return v


_CORRECTOR_SYSTEM_PROMPT = """You are a medical education question editor. A reviewer flagged \
a specific problem with the multiple-choice question below. Make the MINIMAL edit needed to \
fix exactly that problem — do not rewrite parts that weren't flagged.

Respond with ONLY a JSON object of the corrected question, in the form:
{"stem": "...", "correct_answer": "...", "distractors": ["...", "...", "..."], "difficulty": 1}
No prose, no markdown fences, no explanation — JSON only."""


class Corrector:
    def __init__(self, model, processor, device: str):
        self.model = model
        self.processor = processor
        self.device = device

    def correct(self, item: "MCQItem", reason: str, max_retries: int = 2) -> CorrectedMCQ | None:
        user = (
            f"Source passage:\n{item.source_context}\n\n"
            f"Question stem: {item.stem}\n"
            f"Correct answer: {item.correct_answer}\n"
            f"Distractors: {', '.join(item.distractors)}\n"
            f"Stated difficulty tier: {item.triple.difficulty}\n\n"
            f"Reviewer's flagged problem: {reason}"
        )
        last_error: Exception | None = None
        for attempt in range(max_retries + 1):
            raw = self._generate(user, max_new_tokens=512)
            try:
                return self._parse(raw)
            except (ValueError, ValidationError) as e:
                last_error = e
                log.warning("correction parse failed (attempt %d/%d): %s", attempt + 1, max_retries + 1, e)
                log.warning("raw output was: %r", raw)
        log.warning("giving up on correction after %d attempts: %s", max_retries + 1, last_error)
        return None

    def _parse(self, raw: str) -> CorrectedMCQ:
        cleaned = _THINK_BLOCK_RE.sub("", raw)
        matches = _JSON_BLOCK_RE.findall(cleaned)
        if not matches:
            raise ValueError(f"no JSON object found in correction output: {raw[:200]!r}")
        candidate = matches[-1]
        try:
            payload = json.loads(candidate)
        except json.JSONDecodeError:
            payload = json.loads(_repair_truncated_json(candidate))
        return CorrectedMCQ.model_validate(payload)

    def _generate(self, user: str, max_new_tokens: int) -> str:
        messages = [
            {"role": "system", "content": _CORRECTOR_SYSTEM_PROMPT},
            {"role": "user", "content": user},
        ]
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                top_k=None,
            )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


corrector = Corrector(extractor.model, extractor.processor, extractor.device)

## Run critique-and-correct over all generated MCQ items

In [ ]:
@dataclass
class ValidatedMCQItem:
    item: MCQItem
    critique: Critique
    was_corrected: bool


def validate_mcq_items(mcq_items: list[MCQItem], critic: Critic, corrector: Corrector) -> list[ValidatedMCQItem]:
    """One critique-and-correct cycle per item (no iterating until valid). An item that's
    invalid and fails to correct is dropped rather than shipped as a known-bad question."""
    results: list[ValidatedMCQItem] = []
    valid_as_is = corrected = dropped = 0

    for item in tqdm(mcq_items, desc="validating MCQs"):
        critique = critic.critique(item)

        if critique.valid:
            valid_as_is += 1
            results.append(ValidatedMCQItem(item=item, critique=critique, was_corrected=False))
            continue

        reason = critique.reason or "unspecified issue flagged by reviewer"
        fix = corrector.correct(item, reason)
        if fix is None:
            dropped += 1
            continue

        corrected += 1
        fixed_item = MCQItem(
            uid=item.uid,
            doc_id=item.doc_id,
            chunk_id=item.chunk_id,
            topic=item.topic,
            section_path=item.section_path,
            source_context=item.source_context,
            triple=Triple(
                subject=item.triple.subject,
                relation=item.triple.relation,
                object=item.triple.object,
                difficulty=fix.difficulty,
            ),
            stem=fix.stem,
            correct_answer=fix.correct_answer,
            distractors=fix.distractors,
            distractor_sources=["corrected"] * len(fix.distractors),
        )
        results.append(ValidatedMCQItem(item=fixed_item, critique=critique, was_corrected=True))

    print(
        f"validated {len(mcq_items)} item(s): {valid_as_is} valid as-is, "
        f"{corrected} corrected, {dropped} dropped (uncorrectable)"
    )
    return results


validated_mcq_items = validate_mcq_items(mcq_items, critic, corrector)

In [ ]:
# Sanity check — inspect validated MCQ items, flagging what got corrected
for v in validated_mcq_items[:5]:
    item = v.item
    tag = "CORRECTED" if v.was_corrected else "valid"
    print(item.uid, f"[{tag}]", f"(tier {item.triple.difficulty}, topic={item.topic})")
    if v.was_corrected:
        print("  reason:", v.critique.reason)
    print("Q:", item.stem)
    print("  correct:", item.correct_answer)
    for d in item.distractors:
        print("  distractor:", d)
    print("---")

# Step 6 — Output Format and Traceability

Each MCQ is serialized as a JSON object carrying full provenance: the stem, four options with
the correct one flagged, the source chunk/doc ID, the original triple it was generated from,
the difficulty tier, the section path as topic tag, and the critique verdict. Every question
traces back to a specific chunk and triple — evidence of grounded generation, not hallucinated
content.

Questions are grouped by document, then by section path, giving a topic-organized question
bank. Each section path maps directly to a future Neo4j community — questions tagged with a
section path slot straight into the matching community once the graph exists.

## Serialize each MCQ with full provenance

In [ ]:
def serialize_mcq(v: ValidatedMCQItem) -> dict:
    item = v.item
    options = [item.correct_answer, *item.distractors]
    random.shuffle(options)

    return {
        "uid": item.uid,
        "stem": item.stem,
        "options": [{"text": opt, "correct": opt == item.correct_answer} for opt in options],
        "source": {
            "doc_id": item.doc_id,
            "chunk_id": item.chunk_id,
        },
        "triple": {
            "subject": item.triple.subject,
            "relation": item.triple.relation,
            "object": item.triple.object,
        },
        "difficulty": item.triple.difficulty,
        "topic_tag": item.section_path,
        "distractor_sources": item.distractor_sources,
        "critique": {
            "valid_as_generated": not v.was_corrected,
            "reason": v.critique.reason,
        },
    }

## Group by document, then by section path

In [ ]:
def group_by_doc_and_section(validated_mcq_items: list[ValidatedMCQItem]) -> dict:
    """doc_id -> section_path (joined with ' > ') -> list of serialized MCQs. The joined
    section path is a stable string key; the original list form is kept on each question
    under "topic_tag" for when a real section-path structure (Neo4j community) is needed."""
    grouped: dict[str, dict[str, list[dict]]] = defaultdict(lambda: defaultdict(list))
    for v in validated_mcq_items:
        item = v.item
        section_key = " > ".join(item.section_path) if item.section_path else "UNSECTIONED"
        grouped[item.doc_id][section_key].append(serialize_mcq(v))
    return {doc_id: dict(sections) for doc_id, sections in grouped.items()}


question_bank = group_by_doc_and_section(validated_mcq_items)
num_docs = len(question_bank)
num_sections = sum(len(sections) for sections in question_bank.values())
num_questions = sum(len(qs) for sections in question_bank.values() for qs in sections.values())
print(f"question bank: {num_docs} doc(s), {num_sections} section(s), {num_questions} question(s)")

## Dump the question bank to JSON

In [ ]:
OUTPUT_DIR = Path.cwd() / "mcq_output"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "question_bank.json"

OUTPUT_PATH.write_text(json.dumps(question_bank, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"wrote question bank ({num_questions} question(s)) to {OUTPUT_PATH}")

In [ ]:
# Sanity check — inspect the grouped structure and one full serialized question
for doc_id, sections in question_bank.items():
    print(doc_id)
    for section_key, questions in sections.items():
        print(f"  {section_key}: {len(questions)} question(s)")
print()
print(json.dumps(next(iter(next(iter(question_bank.values())).values()))[0], indent=2, ensure_ascii=False))